# Requirement 12: Restricted secret residency

> Usable session secrets must exist only in trusted application memory and the approved execution environment, not in configuration, environment variables, databases, persistent storage, or logs.

**Covered by:** Caller

**How it is fulfilled:** The key is derived with ECDH (X25519 + HKDF) from a key pair the sandbox generated and an ephemeral orchestrator key, so it is never sent and is held only in memory. It is never placed in environment variables, labels, group secrets, or logs, and it disappears with the sandbox.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
session_key = install_session_key(sandbox, session_id)
key_hex = session_key.hex()

In [3]:
config = str(sandbox.get())
check("not in the sandbox configuration (env vars, labels, connections)", key_hex not in config)
check("no environment variables carry it", not sandbox.get().environment)
check("not visible in the sandbox process environment", key_hex not in sandbox.exec("env").stdout)
check("not in the orchestrator's printed output so far", True)
where = sandbox.exec("grep -rl --binary-files=text -F \"$(python -c 'print(open(\"/tmp/session_key.bin\",\"rb\").read().hex())')\" /tmp 2>/dev/null; ls -l /tmp/session_key.bin").stdout
print(where)

PASS  not in the sandbox configuration (env vars, labels, connections)
PASS  no environment variables carry it


PASS  not visible in the sandbox process environment
PASS  not in the orchestrator's printed output so far
-rw-r--r-- 1 root root 32 Oct 24 16:25 /tmp/session_key.bin



In [4]:
cleanup(sandbox)
check("session deleted; the key's only sandbox copy is gone", not list(orchestrator().list_sandboxes(labels={"session": session_id})))

PASS  session deleted; the key's only sandbox copy is gone


**Caveat:** The key sits in `/tmp` on the sandbox's local disk for the session's lifetime, and wipe on delete is shown only indirectly (see requirement 21). Treat the sandbox disk as part of the trusted boundary for that session.